In [ ]:
import getpass
import os
if getpass.getuser() == "peng": # when testing from Jiacheng's work station
    os.environ['pRT_input_data_path'] = "/data2/peng/extracted_spectra_position_A.npy"
import numpy as np
import pymultinest
import pathlib
import pickle
import pandas as pd
import corner
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
#from petitRADTRANS import Radtrans
import petitRADTRANS as prt
from petitRADTRANS.radtrans import Radtrans
from PyAstronomy.pyasl import fastRotBroad, helcorr
from astropy import constants as const
from astropy import units as u
from astropy.coordinates import SkyCoord

from scipy.special import loggamma
from scipy.ndimage import gaussian_filter
from scipy.signal import savgol_filter

from astropy.io import fits

from petitRADTRANS.config import petitradtrans_config_parser
petitradtrans_config_parser.set_input_data_path('/net/lem/data2/pRT3_formatted')


#convert to pRT3 format
'''
from petitRADTRANS.__file_conversion import convert_all
from petitRADTRANS.__file_conversion import _correlated_k_opacities_dat2h5_external_species
from petitRADTRANS.chemistry.prt_molmass import get_species_molar_mass

convert_all(clean=True) # convert all line files to pRT3 format
'''
workpath = '/data2/peng/'




class Target:

    def __init__(self, wl, fl, err, name='dh_tau_b'):
        self.name=name
        self.fullname='DH_Tau_B'
        if len(wl.shape) == 3: # if loading 2D spectra (orders, detectors, pixels)
            self.n_orders= wl.shape[0] # number or orders
            self.n_dets= wl.shape[1] # number of detectors
            
        elif len(wl.shape) ==1: # if loading flattened 1D spectra
            self.n_orders= 5  # hardcoded for now
            self.n_dets= 3    # hardcoded for now

        else:
            print('Error: wavelength/flux/error shape not recognized!')

        self.n_pixels=2048 # number of pixels per detector
        self.K2166=np.array([[[1921.318,1934.583], [1935.543,1948.213], [1949.097,1961.128]],
                            [[1989.978,2003.709], [2004.701,2017.816], [2018.708,2031.165]],
                            [[2063.711,2077.942], [2078.967,2092.559], [2093.479,2106.392]],
                            [[2143.087,2157.855], [2158.914,2173.020], [2173.983,2187.386]],
                            [[2228.786,2244.133], [2245.229,2259.888], [2260.904,2274.835]],
                            [[2321.596,2337.568], [2338.704,2353.961], [2355.035,2369.534]],
                            [[2422.415,2439.061], [2440.243,2456.145], [2457.275,2472.388]]])[::-1]
        
        self.ra="04h29m41.66s"
        self.dec="26d32m56.5s"
        self.JD=59946.042134 + 2.4e6 # observation date in JD        
        self.color='limegreen' # color of retrieval output
        self.wl = wl
        self.fl = fl
        self.err= err

        print(f'Loaded spectrum for target {self.name} with shape wl:{self.wl.shape}, fl:{self.fl.shape}, err:{self.err.shape}')    

    '''    
    def load_spectrum(self):
        self.cwd = os.getcwd()
        file=pathlib.Path(f'{self.cwd}/{self.name}_spectrum.txt') # should be in same folder
        file=np.genfromtxt(file,skip_header=1,delimiter=' ')
        wl=np.reshape(file[:,0],(self.n_orders,self.n_dets,self.n_pixels)) # wavelength
        fl=np.reshape(file[:,1],(self.n_orders,self.n_dets,self.n_pixels)) # flux
        err=np.reshape(file[:,2],(self.n_orders,self.n_dets,self.n_pixels)) # error
        return wl,fl,err
    '''
 
        
    def get_mask_isfinite(self): # for masking out NaN values

        if len(self.wl.shape) ==1: # if loading flattened 1D spectra
            self.mask_isfinite=np.isfinite(self.fl) # only finite pixels
            
            
        
        elif len(self.wl.shape) ==3: # if loading 2D spectra (orders, detectors, pixels)
            self.mask_isfinite=np.empty((self.n_orders,self.n_dets,self.n_pixels),dtype=bool)
            for i in range(self.n_orders):
                for j in range(self.n_dets):
                    mask_ij = np.isfinite(self.fl[i,j]) # only finite pixels
                    self.mask_isfinite[i,j]=mask_ij
        else:
            print('Error: wavelength/flux/error shape not recognized!')
            
        return self.mask_isfinite
    
class Parameters:

    def __init__(self, free_params, constant_params):

        self.params = {} # all parameters + their values
            
        # Separate the prior range from the mathtext label
        self.param_priors, self.param_mathtext = {}, {}
        for key_i, (prior_i, mathtext_i) in free_params.items():
            self.param_priors[key_i]   = prior_i
            self.param_mathtext[key_i] = mathtext_i

        self.param_keys = np.array(list(self.param_priors.keys())) # keys of free parameters
        self.n_params=len(self.param_keys)
        self.ndim = len(self.param_keys) # number of free parameters
        self.free_params=free_params
        self.constant_params=constant_params
        self.params.update(constant_params) # dictionary with constant parameter values
            
    @staticmethod
    def uniform_prior(bounds):
        return lambda x: x*(bounds[1]-bounds[0])+bounds[0]
    
    def __call__(self, cube, ndim=None, nparams=None):
        if (ndim is None) and (nparams is None):
            self.cube_copy = cube
        else:
            self.cube_copy = np.array(cube[:ndim])
        
        for i, key_i in enumerate(self.param_keys):
            
            if key_i not in ["T1","T2","T3","T4"]:  # to not set cube[i] for T1-T4 beforehand, must stay [0,1]
                cube[i] = self.uniform_prior(self.param_priors[key_i])(cube[i]) # cube is vector of length nparams, values [0,1]
            
            # no temperature inversion for isolated objects, so force temperature to increase to avoid weird fluctuations
            if key_i in ["T1","T2","T3","T4"]: # as long as order in dict T0,T1,T2,T3,T4
                cube[i]=self.uniform_prior([cube[i-1]*0.5,cube[i-1]])(cube[i]) # like in Zhang+2021
            
            self.params[key_i] = cube[i] # add free parameter values to parameter dictionary

        return self.cube_copy
    
class Covariance:
     
    def __init__(self, err): 
        self.err = err
        self.cov_reset() # set up covariance matrix
        
    def cov_reset(self): # make diagonal covariance matrix from uncertainties
        self.cov = self.err**2

    def get_logdet(self): # log of determinant
        self.logdet = np.sum(np.log(self.cov)) 
        return self.logdet

    def solve(self, b): # Solve: cov*x = b, for x (x = cov^{-1}*b)
        return 1/self.cov * b # if diagonal matrix, only invert the diagonal
    
class LogLikelihood:

    def __init__(self,retrieval_object,scale_flux=True,scale_err=True,alpha=2,N_phi=1):

        self.d_flux = retrieval_object.data_flux
        self.d_mask = retrieval_object.mask_isfinite
        self.scale_flux   = scale_flux
        self.scale_err    = scale_err
        self.N_d      = self.d_mask.sum() # number of degrees of freedom / valid datapoints
        self.N_params = retrieval_object.n_params
        self.alpha = alpha # from Ruffio+2019
        self.N_phi = N_phi # number of linear scaling parameters
        
    def __call__(self, m_flux, Cov):

        self.ln_L   = 0.0
        self.chi2_0 = 0.0
        self.m_flux_phi = np.nan*np.ones_like(self.d_flux) # scaled model flux

        N_d = self.d_mask.sum() # Number of (valid) data points
        d_flux = self.d_flux[self.d_mask] # data flux
        m_flux = m_flux[self.d_mask] # model flux
        
        if self.scale_flux: # Find the optimal phi-vector to match the observed spectrum
            self.m_flux_phi[self.d_mask],self.phi=self.get_flux_scaling(d_flux, m_flux, Cov)

        residuals_phi = (self.d_flux - self.m_flux_phi) # Residuals wrt scaled model
        inv_cov_0_residuals_phi = Cov.solve(residuals_phi[self.d_mask]) 
        chi2_0 = np.dot(residuals_phi[self.d_mask].T, inv_cov_0_residuals_phi) # Chi-squared for the optimal linear scaling
        logdet_MT_inv_cov_0_M = 0

        if self.scale_flux:
            inv_cov_0_M    = Cov.solve(m_flux) # Covariance matrix of phi
            MT_inv_cov_0_M = np.dot(m_flux.T, inv_cov_0_M)
            logdet_MT_inv_cov_0_M = np.log(MT_inv_cov_0_M) # (log)-determinant of the phi-covariance matrix

        if self.scale_err: 
            self.s2 = self.get_err_scaling(chi2_0, N_d) # Scale variance to maximize log-likelihood
        logdet_cov_0 = Cov.get_logdet()  # Get log of determinant (log prevents over/under-flow)
        self.ln_L += -1/2*(N_d-self.N_phi) * np.log(2*np.pi)+loggamma(1/2*(N_d-self.N_phi+self.alpha-1)) # see Ruffio+2019
        self.ln_L += -1/2*(logdet_cov_0+logdet_MT_inv_cov_0_M+(N_d-self.N_phi+self.alpha-1)*np.log(chi2_0))
        self.chi2_0 += chi2_0
        self.chi2_0_red = self.chi2_0 / self.N_d # Reduced chi-squared (take degrees of freedom into account)

        return self.ln_L

    def get_flux_scaling(self, d_flux, m_flux, cov): 
        # Solve for linear scaling parameter phi: (M^T * cov^-1 * M) * phi = M^T * cov^-1 * d
        lhs = np.dot(m_flux.T, cov.solve(m_flux)) # Left-hand side
        rhs = np.dot(m_flux.T, cov.solve(d_flux)) # Right-hand side
        phi = rhs / lhs # Optimal linear scaling factor
        return np.dot(m_flux, phi), phi # Return scaled model flux + scaling factors

    def get_err_scaling(self, chi_squared_scaled, N):
        s2 = np.sqrt(1/N * chi_squared_scaled)
        return s2 # uncertainty scaling that maximizes log-likelihood
    

class pRT_spectrum:

    def __init__(self,
                 retrieval_object,
                 spectral_resolution=100_000,  
                 contribution=False): # only for plotting atmosphere.contr_em
        
        self.params=retrieval_object.parameters.params
        self.data_wave=retrieval_object.data_wave
        self.target=retrieval_object.target
        self.atmosphere_objects=retrieval_object.atmosphere_objects
        self.coords = SkyCoord(ra=self.target.ra, dec=self.target.dec, frame='icrs')
        self.species=retrieval_object.species
        self.spectral_resolution=spectral_resolution
        self.lbl_opacity_sampling=retrieval_object.lbl_opacity_sampling

        self.n_atm_layers=retrieval_object.n_atm_layers
        self.pressure = retrieval_object.pressure
        self.temperature = self.make_pt() #P-T profile

        self.gravity = 10**self.params['log_g'] 
        self.contribution=contribution

        # free chemistry with defined VMRs
        self.mass_fractions, self.CO, self.FeH = self.free_chemistry(self.species,self.params)
        self.MMW = self.mass_fractions['MMW']
    
    def read_species_info(self,species,info_key):
        species_info = pd.read_csv(os.path.join('species_info.csv'), index_col=0)
        if info_key == 'pRT_name':
            return species_info.loc[species,info_key]
        if info_key == 'mass':
            return species_info.loc[species,info_key]
        if info_key == 'COH':
            return list(species_info.loc[species,['C','O','H']])
        if info_key in ['C','O','H']:
            return species_info.loc[species,info_key]
        if info_key == 'label':
            return species_info.loc[species,'mathtext_name']
    
    def free_chemistry(self,line_species,params):
        species_info = pd.read_csv(os.path.join('species_info.csv'), index_col=0)
        VMR_He = 0.15
        VMR_wo_H2 = 0 + VMR_He  # Total VMR without H2, starting with He
        mass_fractions = {} # Create a dictionary for all used species
        C, O, H = 0, 0, 0

        for species_i in species_info.index:
            line_species_i = self.read_species_info(species_i,'pRT_name')
            mass_i = self.read_species_info(species_i, 'mass')
            COH_i  = self.read_species_info(species_i, 'COH')

            if species_i in ['H2', 'He']:
                continue
            if line_species_i in line_species:
                VMR_i = 10**(params[f'log_{species_i}'])*np.ones(self.n_atm_layers) #  use constant, vertical profile

                # Convert VMR to mass fraction using molecular mass number
                mass_fractions[line_species_i] = mass_i * VMR_i
                VMR_wo_H2 += VMR_i

                # Record C, O, and H bearing species for C/O and metallicity
                C += COH_i[0] * VMR_i
                O += COH_i[1] * VMR_i
                H += COH_i[2] * VMR_i

        # Add the H2 and He abundances
        mass_fractions['He'] = self.read_species_info('He', 'mass')*VMR_He
        mass_fractions['H2'] = self.read_species_info('H2', 'mass')*(1-VMR_wo_H2)
        H += self.read_species_info('H2','H')*(1-VMR_wo_H2) # Add to the H-bearing species
        
        if VMR_wo_H2.any() > 1:
            print('VMR_wo_H2 > 1. Other species are too abundant!')

        MMW = 0 # Compute the mean molecular weight from all species
        for mass_i in mass_fractions.values():
            MMW += mass_i
        MMW *= np.ones(self.n_atm_layers)
        
        for line_species_i in mass_fractions.keys():
            mass_fractions[line_species_i] /= MMW # Turn the molecular masses into mass fractions
        mass_fractions['MMW'] = MMW # pRT requires MMW in mass fractions dictionary
        CO = C/O
        log_CH_solar = 8.46 - 12 # Asplund et al. (2021)
        FeH = np.log10(C/H)-log_CH_solar
        CO = np.nanmean(CO)
        FeH = np.nanmean(FeH)

        if mass_fractions['MMW'].any() < 1.0:
            print('MMW < 1.0! Check mass fractions!')

        return mass_fractions, CO, FeH
    
    def make_spectrum(self):

        print(f"Temperature range: {np.min(self.temperature):.1f} - {np.max(self.temperature):.1f} K")
        print(f"Pressure range: {np.min(self.pressure):.2e} - {np.max(self.pressure):.2e} bar")
        print("Mass fractions:")
        for species, mf in self.mass_fractions.items():
            if species != 'MMW':
                print(f"  {species}: {np.mean(mf):.2e}")


        atmosphere=self.atmosphere_objects

        #  Check if atmosphere object is properly set up
        print(f"Atmosphere species: {atmosphere.line_species}")

        '''
        pRT2 codes
        atmosphere.calculate_flux(Temperatures=self.temperature,
                        mass_fractions=self.mass_fractions,
                        reference_gravity=self.gravity,
                        MMW=self.MMW,
                        contribution=self.contribution)
        '''
        # --pRT3 codes--
        wl, flux, _=atmosphere.calculate_flux(temperatures=self.temperature,
                        mass_fractions=self.mass_fractions,
                        reference_gravity=self.gravity,
                        mean_molar_masses=self.MMW, 
                        return_contribution=True,
                        frequencies_to_wavelengths=True) # returns flux in W/m2/um
        wl *= 1e7 # convert wavelengths from cm to nm

        #wl = const.c.to(u.km/u.s).value/atmosphere.frequencies/1e-9 # mircons, pRT2: freq -- pRT3: frequencies
        #flux=atmosphere.flux/np.median(atmosphere.flux) --pRT2 codes--

        flux /= np.nanmedian(flux) # normalize flux -- pRT3 codes--
        

        print(f"Raw flux range: {np.min(flux):.2e} - {np.max(flux):.2e}")
        print(f'Raw wavelength range: {np.min(wl):.2f} - {np.max(wl):.2f} nm')
        print(f"Flux median: {np.nanmedian(flux):.2e}")

        # RV+bary shifting and rotational broadening
        v_bary, _ = helcorr(obs_long=-70.40, obs_lat=-24.62, obs_alt=2635, # of Cerro Paranal
                        ra2000=self.coords.ra.value,dec2000=self.coords.dec.value,jd=self.target.JD) # https://ssd.jpl.nasa.gov/tools/jdc/#/cd
        
        wl_shifted= wl*(1.0+(self.params['rv']-v_bary)/const.c.to('km/s').value)
        waves_even = np.linspace(np.min(wl), np.max(wl), wl.size) # wavelength array has to be regularly spaced
        new_spec = np.interp(waves_even, wl_shifted, flux)
        spec = fastRotBroad(waves_even, new_spec, 0.5, self.params['vsini']) # limb-darkening coefficient (0-1)   
        spec = self.convolve_to_resolution(waves_even, spec, self.spectral_resolution)
        self.resolution = int(1e6/self.lbl_opacity_sampling)
        flux=self.instr_broadening(waves_even, spec,out_res=self.resolution,in_res=500000)

        if np.all(flux == flux[0]):
            print("WARNING: Flux is constant!")
            print("This suggests an issue with the atmospheric model")

        # Interpolate/rebin onto the data's wavelength grid
        ref_wave = self.data_wave.flatten() # [nm]
        flux = np.interp(ref_wave, waves_even, flux) # pRT wavelengths from cm to nm

        if self.contribution==True:
            contr_em = atmosphere.contr_em # emission contribution
            self.summed_contr = np.nansum(contr_em,axis=1) # sum over all wavelengths

        #combine with telluric template if needed
        


        return flux
            
    # create pressure-temperature profile from 5 temperature knots
    def make_pt(self): 
        self.T_knots = np.array([self.params['T4'],self.params['T3'],self.params['T2'],self.params['T1'],self.params['T0']])
        self.log_P_knots= np.linspace(np.log10(np.min(self.pressure)),np.log10(np.max(self.pressure)),num=len(self.T_knots))
        sort = np.argsort(self.log_P_knots)
        self.temperature = CubicSpline(self.log_P_knots[sort],self.T_knots[sort])(np.log10(self.pressure))
        return self.temperature
    
    def instr_broadening(self, wave, flux, out_res=1e6, in_res=1e6):
        # Delta lambda of resolution element is FWHM of the LSF's standard deviation
        sigma_LSF = np.sqrt(1/out_res**2-1/in_res**2)/(2*np.sqrt(2*np.log(2)))
        spacing = np.mean(2*np.diff(wave) / (wave[1:] + wave[:-1]))
        # Calculate the sigma to be used in the gauss filter in pixels
        sigma_LSF_gauss_filter = sigma_LSF / spacing
        # Apply gaussian filter to broaden with the spectral resolution
        flux_LSF = gaussian_filter(flux, sigma=sigma_LSF_gauss_filter,mode='nearest')
        return flux_LSF
    
    def convolve_to_resolution(self, in_wlen, in_flux, out_res, in_res=None):
        if isinstance(in_wlen, u.Quantity):
            in_wlen = in_wlen.to(u.nm).value
        if in_res is None:
            in_res = np.mean((in_wlen[:-1]/np.diff(in_wlen)))
        # delta lambda of resolution element is FWHM of the LSF's standard deviation:
        sigma_LSF = np.sqrt(1./out_res**2-1./in_res**2)/(2.*np.sqrt(2.*np.log(2.)))
        spacing = np.mean(2.*np.diff(in_wlen)/(in_wlen[1:]+in_wlen[:-1]))
        # Calculate the sigma to be used in the gauss filter in pixels
        sigma_LSF_gauss_filter = sigma_LSF/spacing
        out_flux = np.tile(np.nan, in_flux.shape)
        nans = np.isnan(in_flux)
        out_flux[~nans] = gaussian_filter(in_flux[~nans], sigma = sigma_LSF_gauss_filter,mode = 'reflect')
        return out_flux
    
class Retrieval:

    def __init__(self,parameters,N_live_points,evidence_tolerance, target, testing=True):
        
        self.N_live_points=int(N_live_points) # number of live points
        self.evidence_tolerance=float(evidence_tolerance) # evidence tolerance
        self.target=target
        #self.data_wave,self.data_flux,self.data_err=self.target.load_spectrum()
        self.mask_isfinite=self.target.get_mask_isfinite() # mask nans, shape (orders,detectors)
        self.K2166=self.target.K2166 # wavelength setting
        self.parameters=parameters
        self.species=self.get_species(param_dict=self.parameters.params)
        self.testing=testing
        
        if testing == True:
            ############# for now, let's just do one  order/detector ###########
            self.order= target.n_orders -1  # last order
            self.detector= 1  # middle detector

            self.data_wave=self.target.wl[self.order,self.detector]
            self.data_flux=self.target.fl[self.order,self.detector]
            self.data_err=self.target.err[self.order,self.detector]
            self.mask_isfinite=self.target.mask_isfinite[self.order,self.detector]

            self.data_flux /= np.nanmedian(self.data_flux) #normalize input data flux
        else:
            self.data_wave=self.target.wl
            self.data_flux=self.target.fl
            self.data_err=self.target.err
            self.mask_isfinite=self.target.mask_isfinite

            #self.data_flux /= np.nanmedian(self.data_flux) #normalize input data flux
        ####################################################################
        
        self.n_params = len(parameters.free_params)
        self.output_dir = pathlib.Path(f'{os.getcwd()}/retrievals/N{self.N_live_points}_ev{self.evidence_tolerance}')
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.lbl_opacity_sampling=3
        self.n_atm_layers=50
        self.pressure = np.logspace(-6,2,self.n_atm_layers)  # like in deRegt+2024
        self.Cov = Covariance(err=self.data_err[self.mask_isfinite]) # simple diagonal covariance matrix
        self.LogLike = LogLikelihood(retrieval_object=self,scale_flux=True,scale_err=True)

        # redo atmosphere objects when adding new species
        self.atmosphere_objects=self.get_atmosphere_objects()
        self.callback_label='live_' # label for plots
        self.prefix='pmn_'
        self.color=self.target.color

    def get_species(self,param_dict): # get pRT species name from parameters dict
        species_info = pd.read_csv(os.path.join('species_info.csv'), index_col=0)
        self.chem_species=[]
        for par in param_dict:
            if 'log_' in par and par!='log_g': # get all species in params dict, they are in log, ignore other log values
                self.chem_species.append(par)
        species=[]
        for chemspec in self.chem_species:
            species.append(species_info.loc[chemspec[4:],'pRT_name'])
        return species

    def get_atmosphere_objects(self,redo=True):

        file=pathlib.Path(f'atmosphere_objects.pickle')
        if file.exists() and redo==False:
            with open(file,'rb') as file:
                atmosphere_objects=pickle.load(file)
                return atmosphere_objects
        else:
            print ('Creating new atmosphere objects...')

            wl_pad=7 # wavelength padding because spectrum is not wavelength shifted yet
            if self.testing==True:
             
                wlmin=np.min(self.K2166[self.order])-wl_pad
                wlmax=np.max(self.K2166[self.order])+wl_pad
            else:
                wlmin=np.min(self.data_wave)-wl_pad
                wlmax=np.max(self.data_wave)+wl_pad

            wlen_range=np.array([wlmin,wlmax])*1e-7 # nm to cm

            boundary = wlen_range * 1e4  # cm to micron

            print(f'Wavelength range for atmosphere object: {wlen_range*1e7} nm')

            #check up the following parameters for atmosphere object
            print(f'Line species: {self.species}')
            print(f'Pressure levels: {self.pressure}')
            print(f'Wavelength boundaries: {boundary}')
            print(f'Line-by-line opacity sampling: {self.lbl_opacity_sampling}')


            atmosphere_objects = Radtrans(line_species=self.species,
                                rayleigh_species = ['H2', 'He'],
                                gas_continuum_contributors = ['H2-H2', 'H2-He'],
                                wavelength_boundaries=boundary, 
                                line_opacity_mode='lbl',
                                line_by_line_opacity_sampling=self.lbl_opacity_sampling,
                                pressures=self.pressure) # take every nth point (=3 in deRegt+2024)
            

            #atmosphere_objects.setup_opa_structure(self.pressure)
            with open(file,'wb') as file: # save so that they don't need to be created every time
                pickle.dump(atmosphere_objects,file)
            return atmosphere_objects

    def PMN_lnL(self,cube=None,ndim=None,nparams=None):
        self.model_object=pRT_spectrum(self)
        self.model_flux=self.model_object.make_spectrum()
        ln_L = self.LogLike(self.model_flux, self.Cov) # calcuate log-likelihood
        return ln_L

    def PMN_run(self,N_live_points=None, evidence_tolerance=0.5, resume=True): # run pymultinest
        pymultinest.run(LogLikelihood=self.PMN_lnL,
                        Prior=self.parameters,
                        n_dims=self.parameters.n_params, 
                        outputfiles_basename=f'{self.output_dir}/{self.prefix}', 
                        verbose=True,const_efficiency_mode=True,sampling_efficiency = 0.5,
                        n_live_points=N_live_points,
                        resume=resume, # resume from prevous unfinished run
                        evidence_tolerance=evidence_tolerance, # recommended is 0.5, high number -> finished earlier
                        dump_callback=self.PMN_callback,
                        n_iter_before_update=10) # iterations until calling PMN_callback

    # provides live updates during the retrieval
    def PMN_callback(self,n_samples,n_live,n_params,live_points,posterior, 
                    stats,max_ln_L,ln_Z,ln_Z_err,nullcontext):
        
        print (f'PMN callback at {n_samples} samples, max lnL: {max_ln_L}, lnZ: {ln_Z} +/- {ln_Z_err}')
        self.bestfit_params = posterior[np.argmax(posterior[:,-2]),:-2] # parameters of best-fitting model
        self.posterior = posterior[:,:-2] # remove last 2 columns to get posterior
        self.params_dict,self.model_flux=self.get_params_and_spectrum()
        self.cornerplot() # make cornerplot of live posterior
     
    def PMN_analyse(self):
        # set up PMN analyzer object
        analyzer = pymultinest.Analyzer(n_params=self.parameters.n_params,
                                        outputfiles_basename=f'{self.output_dir}/{self.prefix}')  
        stats = analyzer.get_stats()
        self.posterior = analyzer.get_equal_weighted_posterior() # equally-weighted posterior distribution
        self.posterior = self.posterior[:,:-1] 
        np.save(f'{self.output_dir}/{self.callback_label}posterior.npy',self.posterior)
        self.lnZ = stats['nested importance sampling global log-evidence']

    def get_params_and_spectrum(self): 
        
        # make dictionary of evaluated parameters
        self.params_dict={}
        for i,key in enumerate(self.parameters.param_keys):
            medians = np.array([np.percentile(self.posterior[:,j], [50.0], axis=-1) for j in range(self.posterior.shape[1])])
            self.params_dict[key]=medians[i] # add median of evaluated params

        # create final spectrum
        self.model_object=pRT_spectrum(self)
        self.model_flux=self.model_object.make_spectrum()
        self.params_dict['[Fe/H]']=self.model_object.FeH
        self.params_dict['C/O']=self.model_object.CO

        # get scaling parameters phi and s2 of bestfit model through likelihood
        self.log_likelihood = self.LogLike(self.model_flux, self.Cov)
        self.params_dict['phi']=self.LogLike.phi # flux scaling
        self.params_dict['s2']=self.LogLike.s2 # error scaling
        self.params_dict['chi2']=self.LogLike.chi2_0_red # save reduced chi^2
        if self.callback_label=='final_':
            self.params_dict['lnZ']=self.lnZ # save lnZ
                
        with open(f'{self.output_dir}/{self.callback_label}params_dict.pickle','wb') as file:
            pickle.dump(self.params_dict,file)
        
        return self.params_dict,self.model_flux

    def evaluate(self):
        self.callback_label='final_'
        self.PMN_analyse() # get/save bestfit params and final posterior
        self.params_dict,self.model_flux=self.get_params_and_spectrum() # all params + scaling
        self.cornerplot()

    def run_retrieval(self): 

        print(f'\n ------ Nlive: {self.N_live_points} - ev: {self.evidence_tolerance} ------ \n')
        self.PMN_run(N_live_points=self.N_live_points,evidence_tolerance=self.evidence_tolerance)
        self.evaluate() # creates plots and saves self.params_dict
        print('\n ----------------- Done ---------------- \n')
        
    def cornerplot(self):
        labels=list(self.parameters.param_mathtext.values())
        fontsize=10
        fig = plt.figure(figsize=(self.n_params,self.n_params),dpi=200) # fix size to avoid memory issues
        corner.corner(self.posterior, 
                        labels=labels, 
                        title_kwargs={'fontsize':fontsize},
                        label_kwargs={'fontsize':fontsize},
                        color=self.color,
                        linewidths=0.5,
                        fill_contours=True,
                        quantiles=[0.16,0.5,0.84],
                        title_quantiles=[0.16,0.5,0.84],
                        show_titles=True,
                        fig=fig,
                        quiet=True) # supresses errors
        plt.subplots_adjust(wspace=0,hspace=0)
        plt.rc('xtick',labelsize=fontsize)
        plt.rc('ytick',labelsize=fontsize)     

        #plt.show()
        
        fig.savefig(f'{self.output_dir}/{self.callback_label}cornerplot.pdf',bbox_inches="tight",dpi=200)
        
        plt.close() # to avoid memory issues

In [ ]:
night = '2022-12-31'

spectra_AB_reordered = np.load('/data2/peng/extracted_spectra_position_AB.npy')
spectra_AB_err_reordered=np.load('/data2/peng/extracted_spectra_position_AB_err.npy')

path_wl_cal = workpath + night +'/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits'
wave_hdu = fits.open(path_wl_cal)
wave = np.array(wave_hdu[1].data)[:,0:5,]  # (3, 5, 2048)
# moving axis to (orders, detectors, pixels)
wave_reordered = np.transpose(wave, (1, 0, 2))  # (5, 3, 2048)

#Normalize the spectra before flattening
for order in range(5):
    for det in range(3):
        median_flux = np.nanmedian(spectra_AB_reordered[order, det])
        spectra_AB_reordered[order, det] /= median_flux
        spectra_AB_err_reordered[order, det] /= median_flux

#flatten the spectra to (1, 2048*number of orders*number of detectors)

spectra_AB_reordered = spectra_AB_reordered.reshape(1, -1)
spectra_AB_err_reordered = spectra_AB_err_reordered.reshape(1, -1)
wave_reordered = wave_reordered.reshape(1, -1)

spectra_AB_flat = spectra_AB_reordered.flatten()
spectra_AB_err_flat = spectra_AB_err_reordered.flatten()
wave_flat = wave_reordered.flatten()


#check if flattening is correct
print ('shape check after flattening:')
print('wave:', wave_flat.shape)
print('spectra_AB:', spectra_AB_flat.shape)
print('spectra_AB_err:', spectra_AB_err_flat.shape)

#turn null values to nan and then mask all nans and infs
spectra_AB_flat = np.where(spectra_AB_flat==0, np.nan, spectra_AB_flat)
spectra_AB_err_flat = np.where(spectra_AB_err_flat==0, np.nan, spectra_AB_err_flat)

mask = np.isfinite(spectra_AB_flat) & np.isfinite(spectra_AB_err_flat)

plt.scatter(wave_flat[mask], spectra_AB_flat[mask], s=0.5, alpha=0.7)
plt.show()

#------------set up retriveal parameters----------------#

constant_params = {#'rv': 32,
                   #'log_g': 3.75,
                   #'T0' : 3000, # bottom of the atmosphere (hotter)
                   #'T1' : 2000,
                   #'T2' : 1200,
                   #'T3' : 800,
                   #'T4' : 400, # top of atmosphere (cooler)
                   }

# free parameters we will retrieve - format: key, prior range, mathtext label (for plotting)
free_params = {'rv': ([0,30], r'$v_{\rm rad}$'), # km/s
                'vsini': ([0,40], r'$v$ sin$i$'), # km/s
                'log_g':([3.0,4.0], r'log $g$'),
                'T0' : ([100,4000], r'$T_0$'), # bottom of the atmosphere (hotter)
                'T1' : ([0,4000], r'$T_1$'),
                'T2' : ([0,4000], r'$T_2$'),
                'T3' : ([0,4000], r'$T_3$'),
                'T4' : ([0,4000], r'$T_4$'), # top of atmosphere (cooler)
                'log_H2O':([-12,-1], r'log H$_2$O'), # if free chemistry, define VMRs
                'log_12CO':([-12,-1], r'log $^{12}$CO'),
                'log_13CO':([-12,-1], r'log $^{13}$CO'),
                'log_CH4':([-12,-1], r'log CH$_4$')}

# initialize parameters class object
parameters = Parameters(free_params,constant_params)

# initialize free parameters by randomly drawing from their prior ranges
cube = np.random.rand(parameters.ndim) 
parameters(cube)

species=pd.DataFrame(parameters.param_keys,columns=['species'])
# species.to_csv('species_info.csv',index=False)

# initialize retrieval object
T=Target(wl=wave_flat[mask], fl=spectra_AB_flat[mask], err=spectra_AB_err_flat[mask], name='dh_tau_b')
retrieval=Retrieval(parameters=parameters, N_live_points=200, evidence_tolerance=0.5, target=T, testing=False)
retrieval.run_retrieval()



 

In [ ]:
retrieval.evaluate()
retrieval.get_params_and_spectrum()